# Learning con IBM Logical Neural Networks

Questo notebook dimostra come **IBM LNN può apprendere** pesi logici da dati con eccezioni, ottimizzando "quanto" una regola è affidabile tramite gradient descent.

## Cosa Imparerai

- Training end-to-end di LNN con gradient descent
- Gestione di contraddizioni e dati rumorosi
- Loss function basata su contraddizioni logiche
- Apprendimento di "forza" variabile delle regole

## Scenario: Social Network

Vogliamo apprendere quanto queste regole sociali sono affidabili:
1. "Gli amici di solito hanno gusti simili"
2. "Le persone simili di solito diventano amiche"
3. "L'amico del mio amico è mio amico" (transitività)

Ma ci sono **eccezioni**: gli opposti si attraggono, persone simili ma che non si conoscono, ecc.

---

**Nota sull'architettura**: Questo notebook segue le best practice DRY. La logica è nel file `learning_weights.py`.

## Setup Ambiente

In [ ]:
# Installazione dipendenze
!pip install -q git+https://github.com/IBM/LNN.git
!pip install -q torch>=2.0.0 numpy>=1.24.0 matplotlib>=3.5.0

# Auto-download del modulo per Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("🌐 Esecuzione su Google Colab")
except ImportError:
    IN_COLAB = False
    print("💻 Esecuzione locale")

if IN_COLAB:
    import urllib.request
    url = "https://raw.githubusercontent.com/gianlucamazza/neuro_llm/main/examples/04_learning/learning_weights.py"
    print("📥 Download modulo learning_weights.py...")
    urllib.request.urlretrieve(url, "learning_weights.py")
    print("✓ Modulo scaricato correttamente")
else:
    print("✓ Usando file locale learning_weights.py")

print("\n" + "="*50)
print("Setup completato!")
print("="*50)

## Import Moduli

In [ ]:
from learning_weights import SocialNetworkLearner, generate_synthetic_data, plot_learning_curve
from lnn import Loss

print("✓ SocialNetworkLearner caricato correttamente")
print("✓ Pronto per training LNN!")

## Creazione Sistema e Dataset

Creiamo il sistema di learning e generiamo un dataset sintetico con eccezioni.

In [ ]:
print("="*70)
print("LEARNING CON LNN: Apprendimento Regole Sociali")
print("="*70)

# Crea sistema
learner = SocialNetworkLearner()
print("\n✓ Sistema di learning creato con successo!")

### Generazione Dataset

Il dataset include:
- **Casi normali**: Regole rispettate
- **Eccezioni**: Regole violate (opposti che si attraggono, etc.)

In [ ]:
print("\n[1] Generazione dataset sintetico...")
print("    Include casi normali + eccezioni alle regole")

data = generate_synthetic_data()

# Conta dati
from lnn import Predicate
n_friendships = len([v for v in data[Predicate('Amico', arity=2)].values() if v])
n_similarities = len([v for v in data[Predicate('Simile', arity=2)].values() if v])

print(f"    ✓ Amicizie: {n_friendships}")
print(f"    ✓ Similarità: {n_similarities}")

### Caricamento Dati

In [ ]:
print("\n[2] Caricamento dati in LNN...")
learner.add_training_data(data)
print("✓ Dati caricati nel modello")

## Inferenza Pre-Training

Controlliamo la loss iniziale prima del training.

In [ ]:
print("\n[3] Inferenza PRE-training...")
learner.model.infer()
pre_loss = learner.model.loss(Loss.LOGICAL_CONTRADICTION).item()
print(f"    Loss iniziale (contraddizioni): {pre_loss:.6f}")

## Training

LNN apprende i pesi ottimali minimizzando le contraddizioni logiche.

La **loss function** penalizza:
- Contraddizioni tra fatti e regole
- Incertezze non risolte

Il training è **completamente differenziabile** (gradient descent).

In [ ]:
print("\n[4] Training...")
losses = learner.train(epochs=100, learning_rate=0.01)

## Inferenza Post-Training

Testiamo le predizioni del modello dopo il training.

In [ ]:
print("\n[5] Inferenza POST-training...")
learner.model.infer()
print("✓ Inferenza completata")

## Esempi di Predizioni

Vediamo come il modello gestisce diversi casi.

In [ ]:
print("\n" + "="*70)
print("ESEMPI DI PREDIZIONI DOPO TRAINING")
print("="*70)
print()

test_cases = [
    ('Alice', 'Bob', 'Simile', "Alice e Bob sono amici → dovrebbero essere simili"),
    ('Frank', 'George', 'Simile', "Frank e George sono amici ma OPPOSTI (eccezione)"),
    ('Helen', 'Igor', 'Amico', "Helen e Igor sono simili ma NON amici (no interazione)"),
    ('Alice', 'Charlie', 'Amico', "Alice e Charlie: transitività via Bob"),
]

for p1, p2, pred_name, description in test_cases:
    bounds = learner.predict(p1, p2, pred_name)
    confidence = (bounds[0] + bounds[1]) / 2

    if confidence > 0.7:
        status = "ALTA"
        emoji = "✅"
    elif confidence > 0.4:
        status = "MEDIA"
        emoji = "⚠️"
    else:
        status = "BASSA"
        emoji = "❌"

    print(f"{emoji} {status:5s} | {pred_name}({p1}, {p2})")
    print(f"          Bounds: [{bounds[0]:.3f}, {bounds[1]:.3f}] ~{confidence*100:.1f}%")
    print(f"          {description}")
    print()

## Visualizzazione Learning Curve

Plottiamo la riduzione della loss durante il training.

In [ ]:
import matplotlib.pyplot as plt

print("\n[6] Generazione grafico...")

plt.figure(figsize=(10, 6))
plt.plot(losses, linewidth=2, color='#2E86AB')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (Logical Contradiction)', fontsize=12)
plt.title('LNN Training: Loss Reduction', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n✓ Riduzione loss: da {losses[0]:.4f} a {losses[-1]:.4f} ({(1-losses[-1]/losses[0])*100:.1f}%)")

## Conclusioni

### Caratteristiche dimostrate:

1. **Training end-to-end** con gradient descent su regole logiche
2. **Gestione eccezioni** - regole non assolute ma con "forza" variabile
3. **Loss function logica** - penalizza contraddizioni
4. **Differenziabilità completa** - backpropagation attraverso operazioni logiche

### Come funziona:

- LNN assegna **pesi apprendibili** a operatori logici (AND, OR, IMPLIES)
- Durante training, ottimizza pesi per **minimizzare contraddizioni**
- Regole con eccezioni frequenti ottengono **pesi più bassi**
- Il sistema impara **quanto** fidarsi di ogni regola

### Vantaggi:

- **Vs Rule-based classico**: Gestisce eccezioni e rumore
- **Vs Neural Networks puri**: Mantiene interpretabilità e struttura logica
- **Vs Probabilistic Logic**: Training più efficiente con gradient descent

### Applicazioni:

- Knowledge base refinement da dati rumorosi
- Rule mining da dataset
- Explainable AI con apprendimento
- Reasoning sotto incertezza

### Osservazioni dai Risultati:

- **Alice-Bob**: Alta confidenza (regola rispettata)
- **Frank-George**: Bassa confidenza (eccezione appresa)
- **Helen-Igor**: Bassa confidenza (manca interazione)
- **Transitività**: Confidenza media (regola debole)

Il sistema ha imparato che le regole non sono assolute ma hanno affidabilità variabile.